# UCS420: Cognitive Computing — Assignment 4
## A Cognitive FAQ System Using Pandas (Nova 2.0)

**Roll Number:** 1024170379

Last two digits of roll number: **7, 9**
- digit 7 → category = `["billing", "account", "general"][7 % 3]` = `["billing","account","general"][1]` = **account**
- digit 9 → category = `["billing", "account", "general"][9 % 3]` = `["billing","account","general"][0]` = **billing**

So the personalized entries fall under the **account** and **billing** categories.

## Setup — Import pandas

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', None)
ROLL_NUMBER = "1024170379"


## Q1: Build Your Personalized Knowledge Base

Building a 6-row FAQ DataFrame: 4 fixed entries + 2 personalized entries derived from the roll number digits above (both category = `billing`).

In [ ]:
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# Personalized entries derived from last two digits of roll number 1024170379 -> 7, 9
# digit 7 -> category[7 % 3] = category[1] = "account"
# digit 9 -> category[9 % 3] = category[0] = "billing"
personalized_entries = [
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Settings > Profile > Contact Details to update your registered mobile number.",
     "keywords": "mobile number update profile", "category": "account"},
    {"question": "why was i charged a late payment penalty",
     "answer": "A late fee is added if payment is made after the due date shown on your invoice.",
     "keywords": "penalty late fine overdue", "category": "billing"},
]

faq_entries = fixed_entries + personalized_entries
df = pd.DataFrame(faq_entries)
df


## Q2: Generate and Score a Hypothesis

A scoring function that takes a query string, matches it against each entry's `question` + `keywords`, and returns all matching entries ranked by confidence (score = number of overlapping words between the query and the entry's question+keywords).

In [ ]:
def score_query(query, df):
    """
    Scores every entry in df against the query string.
    Confidence score = number of query words that appear in the entry's
    question + keywords (case-insensitive, whole-word match).
    Returns a DataFrame of matches sorted by score (descending),
    including only entries with score > 0.
    """
    query_words = set(query.lower().split())
    scores = []

    for idx, row in df.iterrows():
        entry_text = (row["question"] + " " + row["keywords"]).lower()
        entry_words = set(entry_text.split())
        overlap = query_words & entry_words
        score = len(overlap)
        scores.append(score)

    result = df.copy()
    result["score"] = scores
    result = result[result["score"] > 0].sort_values(by="score", ascending=False)
    return result.reset_index(drop=True)


# Demo
query = "how do i pay my fee"
matches = score_query(query, df)
print(f"Query: \"{query}\"")
matches


## Q3: `same_category(category_name, df)`

Returns all questions belonging to a given category. Demonstrated using the category of one of the personalized entries from Q1 (**account**).

In [ ]:
def same_category(category_name, df):
    """Returns all questions in df belonging to the given category."""
    return df[df["category"] == category_name][["question", "category"]].reset_index(drop=True)


# Call it using the category of a personalized entry ("account")
account_questions = same_category("account", df)
print("Questions in category: account")
account_questions


## Q4: Add a Keyword and Save to CSV

Pick one entry, ask the user for a new keyword, add it to that entry's keywords, and save the entire updated DataFrame to `<roll_number>_faq_data.csv`.

In [ ]:
# Pick one entry to update — e.g. the first personalized entry
target_question = "how do i update my registered mobile number"

new_keyword = input(f"Enter a new keyword to add to the entry '{target_question}': ").strip()

mask = df["question"] == target_question
df.loc[mask, "keywords"] = df.loc[mask, "keywords"] + " " + new_keyword

csv_filename = f"{ROLL_NUMBER}_faq_data.csv"
df.to_csv(csv_filename, index=False)

print(f"Updated entry keywords: {df.loc[mask, 'keywords'].values[0]}")
print(f"Saved full DataFrame to: {csv_filename}")
df


## Q5: Count of FAQ Entries per Category (groupby)

In [ ]:
category_counts = df.groupby("category").size().reset_index(name="count")
category_counts


## Q6: Handle Ties in Scoring

Modified scoring function: if two or more entries tie for the highest score, **all** tied entries are printed instead of silently picking one.

Demonstrated with:
- A **tying** query — matches both "fee" entries (`what is the annual fee` and `how can i pay the fee`) equally.
- A **non-tying** query — has one clear best match.

In [ ]:
def score_query_with_ties(query, df, verbose=True):
    """
    Same scoring logic as score_query, but explicitly detects ties for the
    top score. If multiple entries share the highest score, all of them
    are printed (not silently reduced to one).
    """
    matches = score_query(query, df)

    if matches.empty:
        if verbose:
            print(f"Query: \"{query}\" -> No matches found.")
        return matches

    top_score = matches["score"].max()
    tied = matches[matches["score"] == top_score]

    if verbose:
        print(f'Query: "{query}"')
        if len(tied) > 1:
            print(f"TIE detected: {len(tied)} entries tied at top score = {top_score}")
            print("Showing all tied entries (not picking just one):")
            display(tied[["question", "answer", "category", "score"]])
        else:
            print(f"Single best match (score = {top_score}):")
            display(tied[["question", "answer", "category", "score"]])
        print()

    return tied


# --- Demo 1: a query that produces a TIE ---
tie_query = "fee"
tie_result = score_query_with_ties(tie_query, df)

# --- Demo 2: a query that does NOT produce a tie ---
no_tie_query = "reset my password"
no_tie_result = score_query_with_ties(no_tie_query, df)
